In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-10: Preference pairs and DPO

## RL for LLMs, Part 2 (RFT alternative methods), session 10 of 14

## What this session teaches

Direct preference optimization, **DPO**, trains a model on pairs. A **pair** is two completions for one prompt, a winner and a loser. There is no reward model and no sampling loop.

For each pair the update raises the winner's tokens and lowers the loser's.

So a rejected completion can teach. This is **contrastive** learning: learning from the difference between two completions set side by side.

A filter keeps the winners and throws the losers away, so it never sees what separates them. A pair keeps both, and what the model learns is exactly that difference.

The size of the push depends on how far the model has already moved toward the winner, compared with a frozen copy of the model it started from.

DPO matches RLHF only under stated assumptions, and only in what it finally learns, not step by step.

It also sees only an ordering, from stored pairs, as a difference, never a level.

The question to carry away: what can a pair teach that a filter cannot, and what can it not?

By the end you can

- say what a preference pair is, and how contrasting a winner with a loser teaches what a filter cannot
- compute the margin and the loss for one pair
- say in what sense DPO equals RLHF, and what its signal cannot see

**Where this session fits: the topics of the series, in order**

<img src="images/rllm10v3/series_strip_9.png" width=88% alt="Strip of the topics of the series with topic 9, preference pairs and DPO, highlighted.">

- this session is topic 9: preference pairs, and learning from the difference between two completions


## Setting and words

**Why this slide?** The first sections use these words; each is said by what it does.

| Word | What it is |
|:--|:--|
| **winner**, **loser** | the better and the worse completion of a pair; papers say chosen and rejected (the opening's "rejected completion" is the loser) |
| **log probability of a completion** | the sum of its tokens' log probabilities; higher means the model finds it likelier |
| **reference model** | a frozen copy of the starting model, as in RLHF (RLLM-2); not RLLM-8's reference, which is what an update copies |
| **filter** | RLLM-9: gives each completion weight one or zero from its reward |

## How we will teach it: the map

**Why this slide?** Four steps, in order. A filter and a pair are the two cases the second step compares.

<img src="images/rllm10v3/rllm10v3_map.png" width=66% alt="Four boxes in a chain, each with its question: Pair, Contrast, Margin, What DPO learns and misses, with a side note that a filter keeps one completion and a pair keeps both.">

- **Pair:** what a pair is, and what DPO does with one
- **Contrast:** what a pair teaches that a filter cannot
- **Margin:** how big the push is, and the loss that gives it
- **What DPO learns and misses:** in what sense it is RLHF, and what its signal cannot see

## Pair: a prompt, a winner and a loser

*You are here: Pair.*

<img src="images/rllm10v3/rllm10v3_bar_1.png" width=55% alt="Section bar with Pair highlighted.">

**Why this slide?** The new kind of data: two completions of one prompt, ranked.

<img src="images/rllm10v3/rllm10v3_pair.png" width=55% alt="A prompt, a winning completion in green and a losing completion in red.">

- a pair says which completion is better, not by how much (RLLM-2)
- pairs come from people, from a labeler model, or from a reward that ranks sampled completions
- **DPO** is one supervised loss on stored pairs: no reward model and no sampling loop

## Pair: up on the winner, down on the loser

**Why this slide?** The mechanism, before any formula.

<img src="images/rllm10v3/rllm10v3_updown.png" width=62% alt="Six tokens of the winner each with weight plus c and six tokens of the loser each with weight minus c.">

- every token of the winner gets weight $+c$, every token of the loser $-c$, with $c>0$
  - how big $c$ is comes in the Margin section
- the same form as RLLM-8: each token's direction gets a weight
- unlike RLLM-9's filter, the loser is pushed down

## Contrast: what a filter throws away, a pair keeps

*You are here: Contrast.*

<img src="images/rllm10v3/rllm10v3_bar_2.png" width=55% alt="Section bar with Contrast highlighted.">

**Why this slide?** The contrast is the point of the session.

<!-- seed-pay: weight-zero-rejected -->
<img src="images/rllm10v3/rllm10v3_filter_vs_pair.png" width=44% alt="The same four completions: the filter gives weights 1, 1, 0, 0; the pair gives plus c to the best and minus c to the worst.">

- From RLLM-9: a dropped completion teaches nothing. A pair changes that
- the pair uses the best and the worst: the winner is raised and the loser is lowered
  - what the model learns is the difference between them

## Contrast: the push lands where the two differ

**Why this slide?** Why the difference is informative, not just two examples.

<img src="images/rllm10v3/rllm10v3_shared_prefix.png" width=60% alt="A winner and a loser sharing their first four tokens, which are grey because plus c and minus c cancel, then diverging tokens in green and red where the push lands.">

- a token both completions share, at the same prefix, gets $+c$ and $-c$ and cancels
- the push lands on the tokens after they part
- a filter cannot see that place: it never has the loser to compare

## Margin: how far the model has moved, against the reference model

*You are here: Margin.*

<img src="images/rllm10v3/rllm10v3_bar_3.png" width=55% alt="Section bar with Margin highlighted.">

**Why this slide?** The size of the push needs a zero point, and the frozen copy gives it.

<img src="images/rllm10v3/rllm10v3_margin.png" width=60% alt="Log probabilities of winner and loser under the reference model and the trained model, then the two log-ratios plus 2 and minus 3 and the margin 0.5.">

- the **log-ratio** of a completion is its log probability under the model minus under the reference model
- the **margin** is $z=\beta\,(\text{winner's log-ratio}-\text{loser's log-ratio})$, and $z=0$ at the start
  - $\beta$ is RLLM-2's KL coefficient: a large $\beta$ keeps the model near the reference model

## Margin: the loss, and why a wrong pair gets the biggest push

**Why this slide?** It links the size of the push to the loss.

<img src="images/rllm10v3/rllm10v3_pair_weight.png" width=50% alt="Curve of the pair weight against the margin, biggest when the margin is negative.">

- the loss is $-\log\sigma(z)$: RLLM-2's Bradley-Terry loss, with the margin as the reward gap
  - it is $\log 2\approx0.693$ at $z=0$ and falls as $z$ grows
- its gradient puts the **pair weight** $c=\beta\,\sigma(-z)$ on each token of the pair, where $\sigma$ is the logistic sigmoid
- $c$ is large when the model gets the pair wrong, and small when it is already right

## Margin: the loss in code

**Why this slide?** This is what you implement.

```python
# F = torch.nn.functional
def dpo_loss(logp_w, logp_l, ref_logp_w, ref_logp_l, beta):
    # each argument: log-probability of a whole completion (summed over its tokens)
    ratio_w = logp_w - ref_logp_w          # log-ratio of the winner
    ratio_l = logp_l - ref_logp_l          # log-ratio of the loser
    z = beta * (ratio_w - ratio_l)         # the margin
    return -F.logsigmoid(z).mean()
```

- the reference model is frozen, so its log probabilities are computed once and cached

## Margin: a worked pair (synthetic numbers, $\beta=0.1$)

**Why this slide?** Numbers make the curve concrete. The reference model's log probabilities are $-40$ for the winner and $-38$ for the loser: it prefers the loser.

| model | log-prob winner | log-prob loser | log-ratio winner | log-ratio loser | margin z | loss | pair weight |
|---|---|---|---|---|---|---|---|
| start: the model equals the reference model | -40 | -38 | +0 | +0 | +0.0 | 0.693 | 0.0500 |
| after training: winner up, loser down | -38 | -41 | +2 | -3 | +0.5 | 0.474 | 0.0378 |
| far right: margin z = +2 | -30 | -48 | +10 | -10 | +2.0 | 0.127 | 0.0119 |
| far wrong: margin z = -2 | -60 | -38 | -20 | +0 | -2.0 | 2.127 | 0.0881 |

- at the start $z=0$, the loss is $\log 2$, and the pair weight is $\beta/2$
- a pair the model gets wrong (weight 0.088) gets far more push than one it gets right (0.012)

## What DPO learns and misses: where the RL went

*You are here: What DPO learns and misses.*

<img src="images/rllm10v3/rllm10v3_bar_4.png" width=55% alt="Section bar with What DPO learns and misses highlighted.">

**Why this slide?** DPO has a reward idea and a KL coefficient but runs no RL loop. So in what sense is it RL?

<img src="images/rllm10v3/dpo_skips_stages.png" width=52% alt="RLHF fits a reward model then runs RL; DPO skips both stages.">

- RLHF fits a reward model, then runs RL
- DPO skips both: a supervised loss on fixed pairs, with no sampling, no reward model, no baseline

## What DPO learns and misses: the bridge from RLHF's objective

**Why this slide?** It says why skipping is allowed, and under what assumptions.

- RLLM-2's objective: $\max_{\pi}\;\mathbb{E}_{y\sim\pi}[\,r(x,y)\,]-\beta\,\mathrm{KL}(\pi\,\Vert\,\pi_{\text{ref}})$
- its best policy: $\pi^*(y\mid x)\;\propto\;\pi_{\text{ref}}(y\mid x)\,e^{r(x,y)/\beta}$
- solve for the reward: $r(x,y)=\beta\log\dfrac{\pi^*(y\mid x)}{\pi_{\text{ref}}(y\mid x)}+\beta\log Z(x)$, where the last term depends only on the prompt
  - $\beta\times$ log-ratio is the paper's **implicit reward**: a reward only up to that prompt term
- in a pair's Bradley-Terry difference the prompt-only term cancels, so no reward model is needed
- it assumes pairs follow Bradley-Terry, and a model that can represent $\pi^*$

## What DPO learns and misses: the same answer, not the same steps

**Why this slide?** "Only in what it finally learns" made exact, set against RLLM-9.

| | **RLLM-9**: filter, then SFT | **DPO** |
|---|---|---|
| RL-style objective | probability of reaching the threshold | reward minus $\beta\times$KL to the reference model |
| What you run | SFT on the kept samples | a classification loss on pairs |
| Equivalence between | **updates**: in expectation, one step | **solutions**: where the loss is minimized |
| Anchored to the start? | no KL term | yes: KL term, coefficient $\beta$ |
| Needs the current model's samples? | yes | no: stored pairs |

- neither is simply "RL" or "just SFT": ask which objective it equals, and in what sense

## What DPO learns and misses: an ordering, at stored pairs, as a difference

*You are here: What DPO learns and misses.*

<img src="images/rllm10v3/rllm10v3_bar_4.png" width=50% alt="Section bar with What DPO learns and misses highlighted.">

**Why this slide?** The limits a user must know.

<img src="images/rllm10v3/dpo_difference_not_level.png" width=44% alt="Log probabilities of the winner and the loser under the reference model and two trained models: in one both fall yet the margin is larger.">

- **ordering only:** how much better the winner is, or whether both are bad, is invisible
- **fixed states:** only the stored completions' prefixes; behavior no pair covers is not pushed on
- **a difference, not a level:** in trained model B both log probabilities fell, yet its margin is larger than A's (Pal et al., 2024)

## What DPO learns and misses: online DPO makes fresh pairs

**Why this slide?** The standard fix for "stored pairs".

<img src="images/rllm10v3/sample_rank_pair_loop.png" width=52% alt="Sample, reward, pair the best against the worst, contrast relative to the frozen reference model, optionally repeat.">

- sample G completions per prompt, score each, pair the best against the worst, apply the DPO loss
- repeating on the newly trained model moves DPO toward on-policy (RLLM-7's staleness dial)
- a prompt whose G completions tie gives no pair

## Placing DPO on RLLM-8's grid

**Why this slide?** The grid grows by one box each session.

<img src="images/rllm10v3/rllm10v3_grid.png" width=50% alt="RLLM-8's grid with this session's DPO box in the others' samples row, reward column; the hybrid box from RLLM-9; RLLM-11 dashed.">

- others' samples: stored pairs, written earlier
- the reward column: a comparison of whole completions supplies the weights (our interpretation)
- how fine: one weight per completion, $+c$ or $-c$ on every token

## Think about

*State a prediction before you work each one.*

1. Online DPO ranks $G=8$ samples per prompt, and one prompt's completions **all earn the same reward**
   - what happens to that prompt? Which of RLLM-9's two filters uses it?
2. **Method Z**: sample 8 completions, a labeler ranks them, DPO on the (best, worst) pair
   - place it by whose samples, where the weights come from, and how fine they are
   - what changes if a filter keeps the **top 2** for SFT (RLLM-9)?
3. Pairs made **once, up front** versus **resampled each round**
   - where does each sit on RLLM-7's staleness dial; what would you measure?
4. The loss sees only a **margin**
   - design a small check of whether the winner's own log probability **rises** during training
   - what would you log?

## What we established, and what it sets up

- **Established**
  - a loser teaches, as the difference from a winner: a pair weight raises the winner's tokens and lowers the loser's
  - the push is biggest when the model gets the pair wrong
  - DPO's solutions match RLHF's, with Bradley-Terry pairs assumed
- **Limit of this session:** an ordering of whole completions, at stored pairs, as a difference: the winner's own probability can fall
- **Sets up RLLM-11, On-policy distillation: the mechanism**
  - DPO only sees a margin at stored pairs. Can we get a dense signal, a number per token, at the model's own states?

## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*](https://arxiv.org/abs/2305.18290): abstract, introduction, section 4 loss
  - [Rasul, Belkada and von Werra, *Fine-tune Llama 2 with DPO*, Hugging Face blog, 2023](https://huggingface.co/blog/dpo-trl): a short worked example of DPO in practice
- **Additional** (5 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


Where the names come from, if you want to read further:

- **DPO**: Rafailov et al. (2023), *Direct Preference Optimization: Your
  Language Model Is Secretly a Reward Model* (NeurIPS 2023)
- **Bradley-Terry**: Bradley and Terry (1952), a model of paired
  comparisons
- **Reward model from pairs, then RL**: Christiano et al. (2017),
  *Deep Reinforcement Learning from Human Preferences*
- **Preferred likelihood can fall**: Pal et al. (2024), *Smaug: Fixing
  Failure Modes of Preference Optimisation with DPO-Positive*
- **Resampling pairs from the current model**: Guo et al. (2024),
  *Direct Language Model Alignment from Online AI Feedback*


### Aside (1 of 3), justification: where "sigmoid of a difference" comes from

*For your own review; we will not work through it in session.*

**Setup.** Items $i, j$ have **strengths** $s_i, s_j$. We model $P(i \succ j)$, the probability that $i$ is preferred (Bradley and Terry, 1952).

1. **Assume** preference probability is proportional to exponentiated
   strength: $P(i \succ j)=\dfrac{e^{s_i}}{e^{s_i}+e^{s_j}}$.
   *(This is the model's one assumption.)*
2. **Divide** by the opposite probability:
   $\dfrac{P(i\succ j)}{P(j\succ i)}=e^{s_i-s_j}$.
   *(The common denominator cancels.)*
3. So the **log-odds** is $s_i-s_j$, the difference of the strengths.
   *(Take the log of step 2.)*
4. **Invert** the log-odds to a probability:
   $P(i\succ j)=\sigma(s_i-s_j)$, with $\sigma(u)=1/(1+e^{-u})$.
   *(Definition of the sigmoid; it is not an extra assumption.)*

**Result.** "Sigmoid of a difference" *is* Bradley-Terry; reward models fit to human pairs use it (Christiano et al., 2017).


### Aside (2 of 3), justification: DPO's loss from the KL-penalized RL objective

*For your own review; we will not work through it in session. The slide "The bridge from RLHF to DPO" gives the idea.*

**Setup.** $\rew(\x,\y)$ is a reward, $\pi_{\text{ref}}$ the frozen
reference model, $\beta>0$ the KL coefficient.

1. The **RL problem**: maximize
   $\Exp{\y\sim\pi}[\,\rew(\x,\y)\,]-\beta\,\KL(\pi\,\Vert\,\pi_{\text{ref}})$
   over policies $\pi$. *(The standard KL-penalized objective.)*
2. Its **optimal policy** is
   $\pi^*(\y\mid \x)=\dfrac{\pi_{\text{ref}}(\y\mid \x)\,e^{\rew(\x,\y)/\beta}}{Z(\x)}$,
   with $Z(\x)$ a normalizer that does not depend on $\y$.
   *(A known closed form for this objective.)*
3. **Solve for the reward**:
   $\rew(\x,\y)=\beta\log\dfrac{\pi^*(\y\mid \x)}{\pi_{\text{ref}}(\y\mid \x)}+\beta\log Z(\x)$.
   *(Take the log of step 2 and rearrange.)*
4. Put this into the **Bradley-Terry difference** $\rew(\x,\y^+)-\rew(\x,\y^-)$ from
   aside 1.
5. The two $\beta\log Z(\x)$ terms are the **same** (same prompt) and **cancel**.
   *(The unknown normalizer is never computed.)*
6. What remains is $z$, with $\pi_\theta$ standing in for $\pi^*$.
   Maximizing the **likelihood of the observed preferences**, $\sigma(z)$,
   is minimizing $-\log\sigma(z)$: the DPO loss.
   *(Maximum likelihood on observed winners.)*

**What this does and does not say.**

- It shows the loss is the **right classifier if** the pairs follow
  Bradley-Terry with *some* reward and the model class can represent
  the optimum
- It says **nothing** about how well plain gradient descent gets there,
  or how the loss behaves when those assumptions fail


### Aside (3 of 3), justification: the pair weight $\beta\,\sigma(-z)$ on a pair

*For your own review; we will not work through it in session.*

1. $\loss=-\log\sigma(z)$. *(The loss.)*
2. $\dfrac{d}{dz}\big[-\log\sigma(z)\big]=-\sigma(-z)$.
   *(Derivative of $-\log\sigma$, using $1-\sigma(z)=\sigma(-z)$.)*
3. $z$ depends on $\theta$ only through
   $\beta\,[\log\pi_\theta(\y^+\mid \x)-\log\pi_\theta(\y^-\mid \x)]$.
   *(The reference-model terms are constants for the parameters.)*
4. So
   $\nabla_\theta\loss=-\beta\,\sigma(-z)\,\big[\nabla_\theta\log\pi_\theta(\y^+\mid \x)-\nabla_\theta\log\pi_\theta(\y^-\mid \x)\big]$.
   *(Chain rule.)*

5. Write $\g^+=\nabla_\theta\log\pi_\theta(\y^+\mid \x)$ and
   $\g^-=\nabla_\theta\log\pi_\theta(\y^-\mid \x)$. A small step on **one pair** is
   $\Delta\theta=\eta\,\beta\,\sigma(-z)\,(\g^+-\g^-)$.
   *(Gradient descent, step size $\eta$.)*
6. To first order, the **margin** $z$ changes by
   $\Delta z=\beta\,(\g^+-\g^-)\cdot\Delta\theta=\eta\,\beta^2\,\sigma(-z)\,\Vert \g^+-\g^-\Vert^2\ge0$.
   *(A squared norm.)*
7. But the **winner alone** changes by
   $\g^+\cdot\Delta\theta\propto\Vert \g^+\Vert^2-\g^+\cdot \g^-$,
   and the **loser alone** by $\g^+\cdot \g^--\Vert \g^-\Vert^2$.
   *(Dot products.)*
8. Either can have the "wrong" sign when $\g^+$ and $\g^-$ are strongly
   aligned and of unequal size, for instance when the two completions
   share most of their tokens.

**Result, in words.**

- The pair gets **one shared pair weight** $\beta\,\sigma(-z)$, large when
  $z<0$ and small when $z>0$
- What holds is **one pair, a small step, first order**: the *margin* grows;
  a batch of pairs or an adaptive optimizer is not covered
- Each log-probability separately can move either way: "a difference, not a level"
